# Train CBM Concept Layer

Run this notebook for the concept bottleneck model task.

Inputs:
- `data/lyrics/final_CBM_input_data.csv` for the small concept-labeled lyric feature table
- `data/lyrics/theme_lyrics_features.csv` for manual concept labels
- `data/lyrics/master_lyrics_features.csv` for the full lyric feature table that needs predicted concepts

Outputs:
- `data/lyrics/concept_vectors.csv` with predicted concept scores for every full lyric row
- `lyrics/model_outputs/concept_layer_model.pt`
- `lyrics/model_outputs/concept_layer_metadata.json`
- `lyrics/model_outputs/concept_layer_cv_metrics.csv`
- `lyrics/model_outputs/concept_label_report.csv`

## 1. Setup

Imports PyTorch and sklearn, mounts Google Drive in Colab, and writes durable model artifacts under `/content/drive/MyDrive/CLARIFY`.

In [ ]:
# Optional in Colab if packages are missing:
# !pip install -q pandas numpy scikit-learn torch

from pathlib import Path
import copy
import json
import re
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

In [ ]:
# Mount Google Drive and save all durable artifacts under MyDrive/CLARIFY.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

DRIVE_ROOT = Path('/content/drive/MyDrive/CLARIFY')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    DRIVE_ROOT,
    Path('/content/DS3-CLARIFY'),
    Path('/content/drive/MyDrive/DS3-CLARIFY'),
]
PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / 'data' / 'lyrics' / 'final_CBM_input_data.csv').exists()),
    DRIVE_ROOT,
)

TRAINING_DATA_DIR = PROJECT_ROOT / 'data' / 'lyrics'
ARTIFACT_LYRIC_DATA_DIR = DRIVE_ROOT / 'data' / 'lyrics'
MODEL_DIR = DRIVE_ROOT / 'lyrics' / 'model_outputs'
ARTIFACT_LYRIC_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_INPUT_PATH = TRAINING_DATA_DIR / 'final_CBM_input_data.csv'
FULL_LYRICS_INPUT_PATH = TRAINING_DATA_DIR / 'master_lyrics_features.csv'
THEME_PATH = TRAINING_DATA_DIR / 'theme_lyrics_features.csv'
CONCEPT_VECTORS_PATH = ARTIFACT_LYRIC_DATA_DIR / 'concept_vectors.csv'
MODEL_PATH = MODEL_DIR / 'concept_layer_model.pt'
METADATA_PATH = MODEL_DIR / 'concept_layer_metadata.json'
METRICS_PATH = MODEL_DIR / 'concept_layer_cv_metrics.csv'
LABEL_REPORT_PATH = MODEL_DIR / 'concept_label_report.csv'

print('Project root:', PROJECT_ROOT)
print('Drive artifact root:', DRIVE_ROOT)
print('CBM labeled input:', FINAL_INPUT_PATH, FINAL_INPUT_PATH.exists())
print('Full lyrics prediction input:', FULL_LYRICS_INPUT_PATH, FULL_LYRICS_INPUT_PATH.exists())
print('Concept labels:', THEME_PATH, THEME_PATH.exists())
print('Concept vectors save to:', CONCEPT_VECTORS_PATH)
print('CBM model artifacts save to:', MODEL_DIR)

## 2. Define Columns and Merge Training Data

Loads the lyric feature table, joins manual concept labels by song title and artist, and counts fully labeled rows.

In [ ]:
IDENTITY_COLUMNS = ["SONG_TITLE", "ARTIST_NAME", "SONG_ID"]

HANDCRAFTED_FEATURE_COLUMNS = [
    "Word_Count",
    "Unique_Word_Count",
    "Repetition_Score",
    "Average_Line_Length",
    "Vocabulary_Diversity",
    "Title_Repetition",
    "Explicit_Word_Count",
    "Sentiment_Score",
    "Positive_Score",
    "Negative_Score",
    "Emotional_Intensity",
]

CONCEPT_COLUMNS = [
    "Love",
    "Heartbreak",
    "Partying",
    "Drugs",
    "Sex",
    "Violence",
    "Self-Empowerment",
    "Success",
    "Hope",
    "Celebration",
    "Coming_Of_Age",
    "Loneliness",
    "Struggle",
    "Friendship",
    "Traveling",
]

def normalize_key(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())

def embedding_sort_key(column):
    match = re.search(r"(\d+)$", column)
    return int(match.group(1)) if match else -1

def get_embedding_columns(df):
    columns = [col for col in df.columns if re.fullmatch(r"BERT_Embedding_\d+", col)]
    return sorted(columns, key=embedding_sort_key)

def require_columns(df, required, name):
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

In [ ]:
final_input = pd.read_csv(FINAL_INPUT_PATH)
themes = pd.read_csv(THEME_PATH)
prediction_input = pd.read_csv(FULL_LYRICS_INPUT_PATH) if FULL_LYRICS_INPUT_PATH.exists() else final_input.copy()

def ensure_song_id(df, prefix):
    df = df.copy()
    if 'SONG_ID' not in df.columns:
        df['SONG_ID'] = [f'{prefix}_{idx:05d}' for idx in range(len(df))]
    return df

final_input = ensure_song_id(final_input, 'cbm_train')
prediction_input = ensure_song_id(prediction_input, 'lyrics_full')

require_columns(final_input, IDENTITY_COLUMNS + HANDCRAFTED_FEATURE_COLUMNS, "final_CBM_input_data.csv")
require_columns(themes, ["Song Title", "Artist"] + CONCEPT_COLUMNS, "theme_lyrics_features.csv")

embedding_columns = get_embedding_columns(final_input)
FEATURE_COLUMNS = HANDCRAFTED_FEATURE_COLUMNS + embedding_columns
require_columns(prediction_input, IDENTITY_COLUMNS + FEATURE_COLUMNS, "master_lyrics_features.csv")

final_input = final_input.copy()
themes = themes.copy()
prediction_input = prediction_input.copy()
final_input["_join_key"] = final_input["SONG_TITLE"].map(normalize_key) + "|" + final_input["ARTIST_NAME"].map(normalize_key)
themes["_join_key"] = themes["Song Title"].map(normalize_key) + "|" + themes["Artist"].map(normalize_key)

training_data = final_input.merge(
    themes[["_join_key"] + CONCEPT_COLUMNS],
    on="_join_key",
    how="left",
    validate="one_to_one",
    indicator=True,
)

unmatched = training_data["_merge"] != "both"
if unmatched.any():
    raise ValueError(training_data.loc[unmatched, IDENTITY_COLUMNS].head())

training_data = training_data[IDENTITY_COLUMNS + FEATURE_COLUMNS + CONCEPT_COLUMNS].copy()
prediction_data = prediction_input[IDENTITY_COLUMNS + FEATURE_COLUMNS].copy()

labeled_mask = training_data[CONCEPT_COLUMNS].notna().all(axis=1)
print("CBM training rows:", len(training_data))
print("Full prediction rows:", len(prediction_data))
print("Feature columns:", len(FEATURE_COLUMNS))
print("Concept columns:", len(CONCEPT_COLUMNS))
print("Fully labeled rows used for training:", int(labeled_mask.sum()))
print("This trains on labeled rows and predicts concepts for the full lyrics feature file.")
training_data.head()

## 3. Split Features and Concepts

Uses only fully labeled concept rows for supervised training. The full lyrics feature table is still used for unsupervised feature compression and for final concept prediction.

In [ ]:
POSITIVE_THRESHOLD = 1.0
BERT_PCA_COMPONENTS = 64
MIN_POSITIVE_EXAMPLES = 20
MIN_NEGATIVE_EXAMPLES = 20

labeled_data = training_data.loc[labeled_mask].reset_index(drop=True)

Y_raw = labeled_data[CONCEPT_COLUMNS].astype(float).to_numpy()
CONCEPT_SCALE = 10.0 if np.nanmax(Y_raw) > 1.0 else 1.0
Y = np.clip(Y_raw / CONCEPT_SCALE, 0.0, 1.0)
Y_binary = (Y_raw >= POSITIVE_THRESHOLD).astype(float)

def clean_numeric_features(df):
    return (
        df.astype(float)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

def fit_feature_preprocessor(reference_features):
    reference_features = clean_numeric_features(reference_features)

    handcrafted_scaler = StandardScaler()
    handcrafted_scaled = handcrafted_scaler.fit_transform(reference_features[HANDCRAFTED_FEATURE_COLUMNS])

    bert_scaler = StandardScaler()
    bert_scaled = bert_scaler.fit_transform(reference_features[embedding_columns])

    pca_components = min(BERT_PCA_COMPONENTS, bert_scaled.shape[0] - 1, bert_scaled.shape[1])
    if pca_components < 1:
        raise ValueError("Not enough BERT embedding columns or rows to build CBM features.")

    bert_pca = PCA(n_components=pca_components, random_state=SEED)
    bert_pca.fit(bert_scaled)

    combined = np.hstack([handcrafted_scaled, bert_pca.transform(bert_scaled)])
    combined_scaler = StandardScaler()
    combined_scaler.fit(combined)

    return {
        "handcrafted_scaler": handcrafted_scaler,
        "bert_scaler": bert_scaler,
        "bert_pca": bert_pca,
        "combined_scaler": combined_scaler,
        "pca_components": pca_components,
        "processed_feature_columns": HANDCRAFTED_FEATURE_COLUMNS + [f"BERT_PCA_{idx}" for idx in range(pca_components)],
    }

def transform_feature_dataframe(features, preprocessor):
    features = clean_numeric_features(features)
    handcrafted_scaled = preprocessor["handcrafted_scaler"].transform(features[HANDCRAFTED_FEATURE_COLUMNS])
    bert_scaled = preprocessor["bert_scaler"].transform(features[embedding_columns])
    bert_pca_values = preprocessor["bert_pca"].transform(bert_scaled)
    combined = np.hstack([handcrafted_scaled, bert_pca_values])
    return preprocessor["combined_scaler"].transform(combined)

# Fit PCA/scalers on the full lyrics feature distribution. This uses no concept labels,
# but makes the tiny labeled set sit in the same feature space as the 3600-song prediction table.
preprocessor_reference = pd.concat(
    [training_data[FEATURE_COLUMNS], prediction_data[FEATURE_COLUMNS]],
    ignore_index=True,
)
FEATURE_PREPROCESSOR = fit_feature_preprocessor(preprocessor_reference)

X_labeled = transform_feature_dataframe(labeled_data[FEATURE_COLUMNS], FEATURE_PREPROCESSOR)
X_all = transform_feature_dataframe(prediction_data[FEATURE_COLUMNS], FEATURE_PREPROCESSOR)

label_report_rows = []
for idx, concept in enumerate(CONCEPT_COLUMNS):
    positives = int(Y_binary[:, idx].sum())
    negatives = int(len(Y_binary) - positives)
    if positives < MIN_POSITIVE_EXAMPLES or negatives < MIN_NEGATIVE_EXAMPLES:
        status = "needs_more_labels"
    else:
        status = "usable_label_coverage"
    label_report_rows.append({
        "Concept": concept,
        "Labeled_Rows": int(len(Y_binary)),
        "Positive_Rows": positives,
        "Negative_Rows": negatives,
        "Positive_Rate": float(positives / len(Y_binary)),
        "Mean_Label": float(Y_raw[:, idx].mean()),
        "Max_Label": float(Y_raw[:, idx].max()),
        "Coverage_Status": status,
    })

label_report = pd.DataFrame(label_report_rows)
label_report.to_csv(LABEL_REPORT_PATH, index=False)

print("Fully labeled rows:", len(labeled_data))
print("Processed CBM input shape:", X_labeled.shape)
print("Full prediction feature shape:", X_all.shape)
print("BERT PCA components:", FEATURE_PREPROCESSOR["pca_components"])
print("Concept score scale:", CONCEPT_SCALE)
print("Saved label coverage report:", LABEL_REPORT_PATH)
label_report

## 4. Train Concept Predictor

Trains a concept encoder with compact hyperparameter search, K-fold validation, class-imbalance weighting, early stopping, and a ridge baseline. This is still the CBM encoder: lyric features in, concept scores out.

In [ ]:
class ConceptEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.25, depth=2):
        super().__init__()
        layers = [
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
        ]

        current_dim = hidden_dim
        for _ in range(max(depth - 1, 0)):
            next_dim = max(current_dim // 2, output_dim * 2)
            layers.extend([
                nn.Linear(current_dim, next_dim),
                nn.ReLU(),
                nn.LayerNorm(next_dim),
                nn.Dropout(dropout),
            ])
            current_dim = next_dim

        layers.append(nn.Linear(current_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

BATCH_SIZE = 16
EPOCHS = 800
TUNING_EPOCHS = 550
FINAL_EPOCH_FLOOR = 150
EARLY_STOPPING_PATIENCE = 90
KFOLD_SPLITS = min(5, len(labeled_data))
TUNING_FOLDS = min(3, KFOLD_SPLITS)
ENSEMBLE_MEMBERS = 5

# This is the "little something" compute budget: a compact MLP search.
# It is intentionally small because the bottleneck is labels, not GPU time.
HYPERPARAM_CONFIGS = [
    {
        "name": "mlp_medium_balanced",
        "hidden_dim": 128,
        "depth": 2,
        "dropout": 0.30,
        "learning_rate": 8e-4,
        "weight_decay": 5e-4,
        "regression_loss_weight": 0.70,
        "classification_loss_weight": 0.30,
    },
    {
        "name": "mlp_wide_regularized",
        "hidden_dim": 256,
        "depth": 2,
        "dropout": 0.40,
        "learning_rate": 5e-4,
        "weight_decay": 1e-3,
        "regression_loss_weight": 0.60,
        "classification_loss_weight": 0.40,
    },
    {
        "name": "mlp_deeper_classifier",
        "hidden_dim": 192,
        "depth": 3,
        "dropout": 0.35,
        "learning_rate": 6e-4,
        "weight_decay": 8e-4,
        "regression_loss_weight": 0.50,
        "classification_loss_weight": 0.50,
    },
    {
        "name": "mlp_small_low_dropout",
        "hidden_dim": 96,
        "depth": 2,
        "dropout": 0.20,
        "learning_rate": 1e-3,
        "weight_decay": 2e-4,
        "regression_loss_weight": 0.75,
        "classification_loss_weight": 0.25,
    },
]

if KFOLD_SPLITS < 2:
    raise ValueError("Need at least two labeled rows to run validation.")

def predict_numpy(model, x_values):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(x_values, dtype=torch.float32).to(DEVICE))
        preds = torch.sigmoid(logits)
    return preds.cpu().numpy()

def make_pos_weight(y_binary_train):
    positives = y_binary_train.sum(axis=0)
    negatives = y_binary_train.shape[0] - positives
    weights = np.divide(
        negatives,
        np.maximum(positives, 1.0),
        out=np.ones_like(negatives, dtype=float),
        where=positives > 0,
    )
    return torch.tensor(np.clip(weights, 1.0, 8.0), dtype=torch.float32).to(DEVICE)

def combined_loss(logits, y_scaled, y_binary_batch, pos_weight, config):
    probs = torch.sigmoid(logits)
    regression_loss = nn.functional.mse_loss(probs, y_scaled)
    classification_loss = nn.functional.binary_cross_entropy_with_logits(
        logits,
        y_binary_batch,
        pos_weight=pos_weight,
    )
    return (
        config["regression_loss_weight"] * regression_loss
        + config["classification_loss_weight"] * classification_loss
    )

def train_concept_model(
    x_train,
    y_train,
    y_binary_train,
    config,
    x_val=None,
    y_val=None,
    y_binary_val=None,
    max_epochs=EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    seed=SEED,
    verbose=False,
):
    torch.manual_seed(seed)
    model = ConceptEncoder(
        input_dim=x_train.shape[1],
        hidden_dim=config["hidden_dim"],
        output_dim=len(CONCEPT_COLUMNS),
        dropout=config["dropout"],
        depth=config["depth"],
    ).to(DEVICE)

    train_ds = TensorDataset(
        torch.tensor(x_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
        torch.tensor(y_binary_train, dtype=torch.float32),
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    pos_weight = make_pos_weight(y_binary_train)

    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    best_val_loss = float("inf")
    stale_epochs = 0

    if x_val is not None:
        x_val_tensor = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
        y_val_tensor = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
        y_binary_val_tensor = torch.tensor(y_binary_val, dtype=torch.float32).to(DEVICE)
    else:
        x_val_tensor = y_val_tensor = y_binary_val_tensor = None

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for xb, yb, ybb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            ybb = ybb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = combined_loss(logits, yb, ybb, pos_weight, config)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        if x_val_tensor is not None:
            model.eval()
            with torch.no_grad():
                val_logits = model(x_val_tensor)
                val_loss = combined_loss(
                    val_logits,
                    y_val_tensor,
                    y_binary_val_tensor,
                    pos_weight,
                    config,
                ).item()

            if val_loss < best_val_loss - 1e-5:
                best_val_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                stale_epochs = 0
            else:
                stale_epochs += 1

            if stale_epochs >= patience:
                break
        else:
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            best_val_loss = float(np.mean(train_losses))

        if verbose and (epoch == 1 or epoch % 100 == 0):
            print(f"epoch {epoch:04d} | loss={np.mean(train_losses):.5f}")

    model.load_state_dict(best_state)
    return model, best_epoch, best_val_loss

def tune_threshold(y_true_score, y_pred_score):
    true_binary = y_true_score >= POSITIVE_THRESHOLD
    if true_binary.sum() == 0 or true_binary.sum() == len(true_binary):
        return float(POSITIVE_THRESHOLD), 0.0

    threshold_grid = np.linspace(0.25, max(CONCEPT_SCALE, 1.0), 80)
    best_threshold = POSITIVE_THRESHOLD
    best_f1 = -1.0
    for threshold in threshold_grid:
        pred_binary = y_pred_score >= threshold
        _, _, f1, _ = precision_recall_fscore_support(
            true_binary,
            pred_binary,
            average="binary",
            zero_division=0,
        )
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    return float(best_threshold), float(best_f1)

def evaluate_oof_predictions(model_name, pred_scaled, tune=True, fixed_thresholds=None):
    metric_rows = []
    thresholds = {}
    for idx, concept in enumerate(CONCEPT_COLUMNS):
        y_true = Y_raw[:, idx]
        y_pred = np.clip(pred_scaled[:, idx], 0.0, CONCEPT_SCALE)

        if tune:
            threshold, tuned_f1 = tune_threshold(y_true, y_pred)
        else:
            threshold = fixed_thresholds.get(concept, POSITIVE_THRESHOLD) if fixed_thresholds else POSITIVE_THRESHOLD
            tuned_f1 = np.nan
        thresholds[concept] = threshold

        true_binary = y_true >= POSITIVE_THRESHOLD
        pred_binary = y_pred >= threshold
        precision, recall, f1, _ = precision_recall_fscore_support(
            true_binary,
            pred_binary,
            average="binary",
            zero_division=0,
        )

        positives = int(true_binary.sum())
        negatives = int(len(true_binary) - positives)
        if positives < MIN_POSITIVE_EXAMPLES or negatives < MIN_NEGATIVE_EXAMPLES:
            reliability = "low_label_coverage"
        elif f1 < 0.50:
            reliability = "weak_validation_f1"
        else:
            reliability = "usable"

        metric_rows.append({
            "Model": model_name,
            "Concept": concept,
            "Positive_Rows": positives,
            "Negative_Rows": negatives,
            "Threshold": threshold,
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "R2": r2_score(y_true, y_pred),
            "Accuracy": accuracy_score(true_binary, pred_binary),
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "Tuned_F1": tuned_f1,
            "Reliability": reliability,
        })

    return pd.DataFrame(metric_rows), thresholds

fold_indices = list(KFold(n_splits=KFOLD_SPLITS, shuffle=True, random_state=SEED).split(X_labeled))
tuning_fold_indices = fold_indices[:TUNING_FOLDS]

tuning_summary_rows = []
tuning_predictions = {}
tuning_epoch_history = {}

for config_idx, config in enumerate(HYPERPARAM_CONFIGS, start=1):
    config_oof_pred = np.zeros_like(Y, dtype=float)
    config_val_mask = np.zeros(len(Y), dtype=bool)
    config_fold_rows = []
    print(f"\nconfig {config_idx}/{len(HYPERPARAM_CONFIGS)}: {config['name']}")

    for fold_idx, (train_idx, val_idx) in enumerate(tuning_fold_indices, start=1):
        fold_model, best_epoch, best_val_loss = train_concept_model(
            X_labeled[train_idx],
            Y[train_idx],
            Y_binary[train_idx],
            config=config,
            x_val=X_labeled[val_idx],
            y_val=Y[val_idx],
            y_binary_val=Y_binary[val_idx],
            max_epochs=TUNING_EPOCHS,
            seed=SEED + config_idx * 100 + fold_idx,
        )
        config_oof_pred[val_idx] = predict_numpy(fold_model, X_labeled[val_idx])
        config_val_mask[val_idx] = True
        config_fold_rows.append({
            "Config": config["name"],
            "Fold": fold_idx,
            "Train_Rows": int(len(train_idx)),
            "Validation_Rows": int(len(val_idx)),
            "Best_Epoch": int(best_epoch),
            "Validation_Loss": float(best_val_loss),
        })
        print(f"  fold {fold_idx}/{TUNING_FOLDS} | best_epoch={best_epoch} | val_loss={best_val_loss:.5f}")

    # Evaluate manually on the covered tuning folds so each config gets the same rows.
    metric_rows = []
    threshold_rows = {}
    covered_y_raw = Y_raw[config_val_mask]
    covered_pred_scaled = np.clip(config_oof_pred[config_val_mask], 0.0, 1.0) * CONCEPT_SCALE
    for idx, concept in enumerate(CONCEPT_COLUMNS):
        y_true = covered_y_raw[:, idx]
        y_pred = covered_pred_scaled[:, idx]
        threshold, _ = tune_threshold(y_true, y_pred)
        true_binary = y_true >= POSITIVE_THRESHOLD
        pred_binary = y_pred >= threshold
        _, _, f1, _ = precision_recall_fscore_support(
            true_binary,
            pred_binary,
            average="binary",
            zero_division=0,
        )
        metric_rows.append({
            "Config": config["name"],
            "Concept": concept,
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "F1": f1,
        })
        threshold_rows[concept] = threshold

    config_metric_df = pd.DataFrame(metric_rows)
    mean_f1 = float(config_metric_df["F1"].mean())
    mean_mae = float(config_metric_df["MAE"].mean())
    mean_epoch = float(np.mean([row["Best_Epoch"] for row in config_fold_rows]))

    tuning_summary_rows.append({
        **config,
        "Mean_F1": mean_f1,
        "Mean_MAE": mean_mae,
        "Mean_RMSE": float(config_metric_df["RMSE"].mean()),
        "Mean_Best_Epoch": mean_epoch,
    })
    tuning_predictions[config["name"]] = {
        "oof_pred": config_oof_pred,
        "val_mask": config_val_mask,
        "metrics": config_metric_df,
        "thresholds": threshold_rows,
    }
    tuning_epoch_history[config["name"]] = config_fold_rows
    print(f"  summary | mean_f1={mean_f1:.4f} | mean_mae={mean_mae:.4f}")

tuning_summary = (
    pd.DataFrame(tuning_summary_rows)
    .sort_values(["Mean_F1", "Mean_MAE"], ascending=[False, True])
    .reset_index(drop=True)
)
BEST_CONFIG = {
    key: tuning_summary.loc[0, key]
    for key in HYPERPARAM_CONFIGS[0].keys()
}
BEST_CONFIG["hidden_dim"] = int(BEST_CONFIG["hidden_dim"])
BEST_CONFIG["depth"] = int(BEST_CONFIG["depth"])
BEST_CONFIG_NAME = BEST_CONFIG["name"]

print("\nBest CBM config:", BEST_CONFIG_NAME)
try:
    display(tuning_summary.round(4))
except NameError:
    print(tuning_summary.round(4).to_string(index=False))

oof_pred = np.zeros_like(Y, dtype=float)
fold_rows = []
fold_best_epochs = []

print(f"\nFull {KFOLD_SPLITS}-fold validation for best config: {BEST_CONFIG_NAME}")
for fold_idx, (train_idx, val_idx) in enumerate(fold_indices, start=1):
    fold_model, best_epoch, best_val_loss = train_concept_model(
        X_labeled[train_idx],
        Y[train_idx],
        Y_binary[train_idx],
        config=BEST_CONFIG,
        x_val=X_labeled[val_idx],
        y_val=Y[val_idx],
        y_binary_val=Y_binary[val_idx],
        max_epochs=EPOCHS,
        seed=SEED + fold_idx,
    )
    oof_pred[val_idx] = predict_numpy(fold_model, X_labeled[val_idx])
    fold_best_epochs.append(best_epoch)
    fold_rows.append({
        "Config": BEST_CONFIG_NAME,
        "Fold": fold_idx,
        "Train_Rows": int(len(train_idx)),
        "Validation_Rows": int(len(val_idx)),
        "Best_Epoch": int(best_epoch),
        "Validation_Loss": float(best_val_loss),
    })
    print(f"fold {fold_idx}/{KFOLD_SPLITS} | best_epoch={best_epoch} | val_loss={best_val_loss:.5f}")

ridge_oof_pred = np.zeros_like(Y, dtype=float)
for train_idx, val_idx in fold_indices:
    ridge = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0])
    ridge.fit(X_labeled[train_idx], Y[train_idx])
    ridge_oof_pred[val_idx] = np.clip(ridge.predict(X_labeled[val_idx]), 0.0, 1.0)

fold_metrics = pd.DataFrame(fold_rows)
fold_metrics

## 5. Evaluate Concept Prediction

Reports out-of-fold metrics for the tuned CBM, compares against a ridge baseline, and flags concepts that need more manual labels.

In [ ]:
mlp_metrics, concept_thresholds = evaluate_oof_predictions(
    f"cbm_mlp_tuned::{BEST_CONFIG_NAME}",
    np.clip(oof_pred, 0.0, 1.0) * CONCEPT_SCALE,
    tune=True,
)
ridge_metrics, ridge_thresholds = evaluate_oof_predictions(
    "ridge_baseline",
    np.clip(ridge_oof_pred, 0.0, 1.0) * CONCEPT_SCALE,
    tune=True,
)

tuning_detail_metrics = pd.concat(
    [payload["metrics"].assign(Model=name) for name, payload in tuning_predictions.items()],
    ignore_index=True,
)
tuning_detail_metrics = tuning_detail_metrics.rename(columns={"Config": "Tuning_Config"})

metrics = pd.concat([mlp_metrics, ridge_metrics], ignore_index=True)
metrics.to_csv(METRICS_PATH, index=False)
tuning_detail_metrics.to_csv(MODEL_DIR / "concept_layer_hyperparam_metrics.csv", index=False)

comparison = (
    metrics.groupby("Model")
    .agg(
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Mean_F1=("F1", "mean"),
        Usable_Concepts=("Reliability", lambda values: int((values == "usable").sum())),
    )
    .reset_index()
    .sort_values(["Mean_F1", "Mean_MAE"], ascending=[False, True])
)

print("Saved CV metrics:", METRICS_PATH)
print("Saved hyperparameter metrics:", MODEL_DIR / "concept_layer_hyperparam_metrics.csv")
print("Best validation summary:")
try:
    display(comparison.round(4))
    display(mlp_metrics.round(4))
except NameError:
    print(comparison.round(4).to_string(index=False))
    print(mlp_metrics.round(4).to_string(index=False))

## 6. Train Final Concept Encoder

Retrains multiple copies of the best CBM configuration on every fully labeled row, then averages their predictions into one ensemble concept vector per full lyric row.

In [ ]:
FINAL_EPOCHS = int(max(FINAL_EPOCH_FLOOR, np.median(fold_best_epochs)))
print("Best config:", BEST_CONFIG_NAME)
print("Final training epochs per ensemble member:", FINAL_EPOCHS)
print("Ensemble members:", ENSEMBLE_MEMBERS)

final_models = []
all_member_preds = []
final_training_rows = []

for member_idx in range(ENSEMBLE_MEMBERS):
    member_seed = SEED + 1000 + member_idx
    final_model, final_best_epoch, final_loss = train_concept_model(
        X_labeled,
        Y,
        Y_binary,
        config=BEST_CONFIG,
        x_val=None,
        y_val=None,
        y_binary_val=None,
        max_epochs=FINAL_EPOCHS,
        patience=FINAL_EPOCHS + 1,
        seed=member_seed,
        verbose=(member_idx == 0),
    )
    final_models.append(final_model)
    all_member_preds.append(predict_numpy(final_model, X_all))
    final_training_rows.append({
        "Member": int(member_idx + 1),
        "Seed": int(member_seed),
        "Epochs": int(final_best_epoch),
        "Training_Loss": float(final_loss),
    })
    print(f"ensemble member {member_idx + 1}/{ENSEMBLE_MEMBERS} | seed={member_seed} | loss={final_loss:.5f}")

all_pred_scaled = np.stack(all_member_preds, axis=0)
all_pred = np.clip(all_pred_scaled.mean(axis=0) * CONCEPT_SCALE, 0.0, CONCEPT_SCALE)
all_pred_std = all_pred_scaled.std(axis=0) * CONCEPT_SCALE

concept_vector_columns = [f"Concept_{concept}" for concept in CONCEPT_COLUMNS]
concept_uncertainty_columns = [f"ConceptUncertainty_{concept}" for concept in CONCEPT_COLUMNS]
concept_vectors = pd.concat(
    [
        prediction_data[IDENTITY_COLUMNS].reset_index(drop=True),
        pd.DataFrame(all_pred, columns=concept_vector_columns),
        pd.DataFrame(all_pred_std, columns=concept_uncertainty_columns),
    ],
    axis=1,
)
concept_vectors.to_csv(CONCEPT_VECTORS_PATH, index=False)
print("Saved full ensemble concept vectors:", CONCEPT_VECTORS_PATH)
print("Concept-vector rows:", len(concept_vectors))
concept_vectors.head()

## 7. Save Model Artifacts

Saves the ensemble PyTorch model states, preprocessing arrays, tuned thresholds, hyperparameter search results, validation metrics, and run metadata needed by the downstream fusion model or backend.

In [ ]:
def to_jsonable(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")

torch.save(
    {
        "model_state_dict": final_models[0].state_dict(),
        "ensemble_state_dicts": [model.state_dict() for model in final_models],
        "feature_columns": FEATURE_COLUMNS,
        "processed_feature_columns": FEATURE_PREPROCESSOR["processed_feature_columns"],
        "handcrafted_feature_columns": HANDCRAFTED_FEATURE_COLUMNS,
        "embedding_columns": embedding_columns,
        "concept_columns": CONCEPT_COLUMNS,
        "concept_scale": CONCEPT_SCALE,
        "concept_thresholds": concept_thresholds,
        "handcrafted_scaler_mean": FEATURE_PREPROCESSOR["handcrafted_scaler"].mean_,
        "handcrafted_scaler_scale": FEATURE_PREPROCESSOR["handcrafted_scaler"].scale_,
        "bert_scaler_mean": FEATURE_PREPROCESSOR["bert_scaler"].mean_,
        "bert_scaler_scale": FEATURE_PREPROCESSOR["bert_scaler"].scale_,
        "bert_pca_components": FEATURE_PREPROCESSOR["bert_pca"].components_,
        "bert_pca_mean": FEATURE_PREPROCESSOR["bert_pca"].mean_,
        "bert_pca_explained_variance_ratio": FEATURE_PREPROCESSOR["bert_pca"].explained_variance_ratio_,
        "combined_scaler_mean": FEATURE_PREPROCESSOR["combined_scaler"].mean_,
        "combined_scaler_scale": FEATURE_PREPROCESSOR["combined_scaler"].scale_,
        "best_config": BEST_CONFIG,
        "ensemble_members": ENSEMBLE_MEMBERS,
        "model_architecture": "mlp_with_logits_tuned_ensemble",
    },
    MODEL_PATH,
)

metadata = {
    "model_type": "pytorch_mlp_concept_layer_tuned_ensemble",
    "rows": int(len(training_data)),
    "labeled_rows": int(labeled_mask.sum()),
    "full_prediction_rows": int(len(prediction_data)),
    "input_features_raw": len(FEATURE_COLUMNS),
    "input_features_processed": int(X_labeled.shape[1]),
    "handcrafted_features": len(HANDCRAFTED_FEATURE_COLUMNS),
    "bert_embedding_features": len(embedding_columns),
    "bert_pca_components": int(FEATURE_PREPROCESSOR["pca_components"]),
    "concept_outputs": len(CONCEPT_COLUMNS),
    "epochs_requested": EPOCHS,
    "tuning_epochs_requested": TUNING_EPOCHS,
    "final_epochs": FINAL_EPOCHS,
    "ensemble_members": ENSEMBLE_MEMBERS,
    "best_config": BEST_CONFIG,
    "hyperparameter_configs": HYPERPARAM_CONFIGS,
    "hyperparameter_summary": tuning_summary.round(6).to_dict(orient="records"),
    "hyperparameter_fold_metrics": tuning_epoch_history,
    "concept_scale": CONCEPT_SCALE,
    "positive_threshold": POSITIVE_THRESHOLD,
    "concept_thresholds": concept_thresholds,
    "feature_columns": FEATURE_COLUMNS,
    "processed_feature_columns": FEATURE_PREPROCESSOR["processed_feature_columns"],
    "concept_columns": CONCEPT_COLUMNS,
    "cv_folds": int(KFOLD_SPLITS),
    "tuning_folds": int(TUNING_FOLDS),
    "fold_metrics": fold_metrics.to_dict(orient="records"),
    "final_training": final_training_rows,
    "validation_comparison": comparison.round(6).to_dict(orient="records"),
    "cbm_validation_metrics": mlp_metrics.round(6).to_dict(orient="records"),
    "label_report": label_report.to_dict(orient="records"),
    "artifact_paths": {
        "concept_vectors": str(CONCEPT_VECTORS_PATH),
        "model": str(MODEL_PATH),
        "metadata": str(METADATA_PATH),
        "cv_metrics": str(METRICS_PATH),
        "label_report": str(LABEL_REPORT_PATH),
        "hyperparam_metrics": str(MODEL_DIR / "concept_layer_hyperparam_metrics.csv"),
    },
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2, default=to_jsonable), encoding="utf-8")

print("Saved concept vectors:", CONCEPT_VECTORS_PATH)
print("Saved model:", MODEL_PATH)
print("Saved metadata:", METADATA_PATH)
print("Saved CV metrics:", METRICS_PATH)
print("Saved label report:", LABEL_REPORT_PATH)
print("Saved hyperparameter metrics:", MODEL_DIR / "concept_layer_hyperparam_metrics.csv")